In [1]:
import matplotlib.pyplot as plt
import matplotlib.animation as animation

# Configuración de la figura y ejes
fig, ax = plt.subplots(figsize=(10, 6.5))

tasks = ['Secuencial', 'Concurrente (1 Núcleo)', 'Paralelo (2 Núcleos)']

# Bloques de tiempo: (inicio, duración, color, etiqueta)
seq_blocks = [(0, 4, '#4C72B0', 'T1'), (4, 8, '#DD8452', 'T2')]

# Concurrente: Intercalado alternado en tiempo (Quantum de 1 unidad)
conc_t1 = [(0, 1), (2, 3), (4, 5), (6, 7)]  # Tarea 1
conc_t2 = [(1, 2), (3, 4), (5, 6), (7, 8)]  # Tarea 2

# Puntos exactos donde ocurren cambios de contexto
context_switches = [1, 2, 3, 4, 5, 6, 7]

# Paralelo: Ejecución simultánea exacta de 0 a 4
par_blocks = [(0, 4, '#4C72B0', 'T1 (Core 1)'), (0, 4, '#DD8452', 'T2 (Core 2)')]

def update(frame):
    ax.clear()
    ax.set_yticks([1, 2, 3])
    ax.set_yticklabels(tasks, fontsize=11, fontweight='bold')
    ax.set_xlim(0, 10)
    ax.set_ylim(0.4, 3.7)
    ax.set_xlabel('Tiempo (unidades)', fontsize=11, fontweight='bold')
    ax.set_title('Flujo Temporal: Secuencial vs Concurrente vs Paralelo', fontsize=13, pad=15, fontweight='bold')
    ax.grid(axis='x', linestyle='--', alpha=0.5)

    t = frame / 10.0  # Tiempo simulado

    # 1. SECUENCIAL (y = 1)
    for start, dur, color, _ in seq_blocks:
        if t > start:
            current_dur = min(t - start, dur)
            ax.barh(1, current_dur, left=start, height=0.35, color=color, edgecolor='black', alpha=0.85)

    # 2. CONCURRENTE (y = 2): Dividido con marcas de Context Switch
    # Dibujar líneas punteadas verticales en los instantes de conmutación
    for cs in context_switches:
        if t >= cs:
            ax.vlines(x=cs, ymin=1.65, ymax=2.35, colors='#C44E52', linestyles=':', linewidth=1.5, alpha=0.8)
            ax.plot(cs, 2.0, marker='o', markersize=5, color='#C44E52')
            if cs == 1 and t < 3:
                ax.text(cs, 2.4, 'Context\nSwitch', color='#C44E52', fontsize=8, ha='center', fontweight='bold')

    # Sub-línea superior (T1)
    for start, end in conc_t1:
        dur = end - start
        if t > start:
            current_dur = min(t - start, dur)
            ax.barh(2.15, current_dur, left=start, height=0.18, color='#4C72B0', edgecolor='black', alpha=0.85)

    # Sub-línea inferior (T2)
    for start, end in conc_t2:
        dur = end - start
        if t > start:
            current_dur = min(t - start, dur)
            ax.barh(1.85, current_dur, left=start, height=0.18, color='#DD8452', edgecolor='black', alpha=0.85)

    # 3. PARALELO (y = 3): Múltiples núcleos simultáneos
    for start, dur, color, label in par_blocks:
        if t > start:
            current_dur = min(t - start, dur)
            offset = 0.12 if 'Core 1' in label else -0.12
            ax.barh(3 + offset, current_dur, left=start, height=0.18, color=color, edgecolor='black', alpha=0.85)

    # Leyenda explicativa
    ax.text(8.2, 2.2, '■ Tarea 1', color='#4C72B0', fontweight='bold', va='center', fontsize=9)
    ax.text(8.2, 1.8, '■ Tarea 2', color='#DD8452', fontweight='bold', va='center', fontsize=9)
    ax.text(8.2, 2.5, '┆ Context Switch', color='#C44E52', fontweight='bold', va='center', fontsize=8)

# Crear la animación
ani = animation.FuncAnimation(fig, update, frames=85, interval=100)

# Guardar como GIF animado
ani.save('images/ejecucion_comparativa_2.gif', writer='pillow', fps=10)
plt.close()